In [3]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, KFold
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from imblearn.under_sampling import RandomUnderSampler
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             balanced_accuracy_score, confusion_matrix,
                             matthews_corrcoef, cohen_kappa_score, log_loss,
                             mean_squared_error, mean_absolute_error, r2_score)
import time
import os
import skfuzzy as fuzz
from skfuzzy import control as ctrl
 
# Set the number of K folds as a global variable
K_FOLDS = 2
 
# Read the dataset from CSV file
df = pd.read_csv('../../../datasets/dataset-2.csv')
 
# Take 20% of the data
df = df.sample(frac=0.2, random_state=42)
 
# Create a class for fuzzy logic classifier
class FuzzyLogicClassifier:
    def __init__(self):
        self.fuzzy_systems = {}
        self.rules = {}
        self.output_ranges = {}
        self.trained = False
        
    def _create_fuzzy_system(self, X, y):
        # Create universe variables
        # We'll use the first 4 features for the fuzzy system
        num_features = min(4, X.shape[1])
        
        # Create antecedents for each feature
        antecedents = []
        for i in range(num_features):
            feature_name = f'feature_{i}'
            antecedents.append(ctrl.Antecedent(np.linspace(0, X[:, i].max() + 0.01, 100), feature_name))
        
        # Output variable
        # Assuming label is a numeric value
        attack_class = ctrl.Consequent(np.linspace(1, 15, 100), 'attack_class')
        
        # Auto-membership functions
        for i, antecedent in enumerate(antecedents):
            if i == num_features - 1:  # Last feature with only 2 categories
                antecedent.automf(2, names=['low', 'high'])
            else:
                antecedent.automf(3, names=['low', 'medium', 'high'])
        
        # Attack class membership functions
        # Create membership functions for each class (adjust as needed)
        labels = np.unique(y)
        num_classes = len(labels)
        
        # Create membership functions for classes
        for i, label in enumerate(labels):
            if num_classes <= 5:
                # For fewer classes, use dedicated membership functions
                attack_class[f'class_{int(label)}'] = fuzz.trimf(attack_class.universe, 
                    [max(1, label-2), label, min(15, label+2)])
            else:
                # For many classes, group them
                if label <= 5:
                    attack_class[f'low_{int(label)}'] = fuzz.trimf(attack_class.universe, [1, 3, 5])
                elif 5 < label <= 10:
                    attack_class[f'mid_{int(label)}'] = fuzz.trimf(attack_class.universe, [5, 8, 10])
                else:
                    attack_class[f'high_{int(label)}'] = fuzz.trimf(attack_class.universe, [10, 13, 15])
        
        # Define fuzzy rules
        rules = []
        
        # Basic rules
        if num_features >= 2:
            # Get the appropriate membership functions for each class
            low_class = self._get_membership_function_name(attack_class, 6)
            mid_class = self._get_membership_function_name(attack_class, 10)
            high_class = self._get_membership_function_name(attack_class, 13)
            very_high_class = self._get_membership_function_name(attack_class, 14)
            
            rules.append(ctrl.Rule(antecedents[0]['high'] & antecedents[1]['high'], attack_class[mid_class]))
            rules.append(ctrl.Rule(antecedents[0]['low'] & antecedents[1]['low'], attack_class[low_class]))
        
        if num_features >= 3:
            rules.append(ctrl.Rule(antecedents[2]['high'] & antecedents[0]['low'], attack_class[high_class]))
            rules.append(ctrl.Rule(antecedents[2]['low'] & antecedents[1]['high'], attack_class[mid_class]))
        
        if num_features >= 4:
            rules.append(ctrl.Rule(antecedents[3]['high'] & antecedents[0]['high'], attack_class[very_high_class]))
            rules.append(ctrl.Rule(antecedents[3]['low'] & antecedents[1]['medium'], attack_class[mid_class]))
        
        # Additional rules
        if num_features >= 2:
            rules.append(ctrl.Rule(antecedents[0]['medium'] & antecedents[1]['medium'], attack_class[mid_class]))
            rules.append(ctrl.Rule(antecedents[0]['medium'] & antecedents[1]['high'], attack_class[high_class]))
            rules.append(ctrl.Rule(antecedents[0]['low'] & antecedents[1]['low'], attack_class[low_class]))
        
        # Create a control system
        attack_ctrl = ctrl.ControlSystem(rules)
        
        return ctrl.ControlSystemSimulation(attack_ctrl), antecedents
    
    def _get_membership_function_name(self, attack_class, target_class):
        """Find the appropriate membership function name for a given class"""
        # First try exact match
        if f'class_{target_class}' in attack_class.terms:
            return f'class_{target_class}'
        
        # For grouped classes, find the closest match
        if target_class <= 5:
            for key in attack_class.terms:
                if key.startswith('low_'):
                    return key
        elif 5 < target_class <= 10:
            for key in attack_class.terms:
                if key.startswith('mid_'):
                    return key
        else:
            for key in attack_class.terms:
                if key.startswith('high_'):
                    return key
        
        # If no match found, return the first available term
        return list(attack_class.terms.keys())[0]
        
    def fit(self, X, y):
        # Select a subset of features - use the first 4 features
        num_features = min(4, X.shape[1])
        X_subset = X[:, :num_features]
            
        self.fuzzy_system, self.antecedents = self._create_fuzzy_system(X_subset, y)
        self.trained = True
        self.num_features = num_features
        
        # Save the range of output values for normalization
        self.output_min = 1
        self.output_max = 15
        
        # Store unique labels for prediction
        self.unique_labels = np.unique(y)
        
        return self
        
    def predict(self, X):
        if not self.trained:
            raise ValueError("The model has not been trained yet.")
            
        # Select a subset of features - use the first 4 features
        num_features = min(self.num_features, X.shape[1])
        X_subset = X[:, :num_features]
            
        predictions = []
        for i in range(X_subset.shape[0]):
            # Set inputs for each feature
            for j in range(num_features):
                self.fuzzy_system.input[f'feature_{j}'] = X_subset[i, j]
            
            try:
                self.fuzzy_system.compute()
                # Get the output
                attack_class_value = self.fuzzy_system.output['attack_class']
                
                # Find the nearest class in our unique labels
                pred_class = self.unique_labels[np.abs(self.unique_labels - attack_class_value).argmin()]
                
                predictions.append(pred_class)
            except Exception as e:
                # If there's an error, default to the most common class
                # Use the middle value from unique labels as a fallback
                predictions.append(self.unique_labels[len(self.unique_labels)//2])
                
        return np.array(predictions)
 
# Prepare the data
# Extract X and y from the dataset
X = df.iloc[:, :-1].values  # All columns except the last one
y = df.iloc[:, -1].values   # Last column is the label

timing_results = []
 
# Define classifiers and metrics
classifiers = {
    'DecisionTree': DecisionTreeClassifier(),
    'KNN': KNeighborsClassifier(),
    'LogisticRegression': LogisticRegression(max_iter=1000),
    'FuzzyLogic': FuzzyLogicClassifier()
}

# Store results
results = []
 
kf = KFold(n_splits=K_FOLDS, shuffle=True, random_state=42)
 
# Create a directory to save confusion matrices
os.makedirs("confusion_matrices", exist_ok=True)
 
# Helper function for confusion matrix metrics
def confusion_matrix_metrics(cm, classes):
    metrics = {}
    for idx, class_label in enumerate(classes):
        TP = cm[idx, idx]  # True Positives for this class
        FP = cm[:, idx].sum() - TP  # False Positives for this class
        FN = cm[idx, :].sum() - TP  # False Negatives for this class
        TN = cm.sum() - (TP + FP + FN)  # True Negatives for this class
 
        metrics[class_label] = {
            'TPR': TP / (TP + FN + 1e-10) if (TP + FN) > 0 else 0,
            'TNR': TN / (TN + FP + 1e-10) if (TN + FP) > 0 else 0,
            'FPR': FP / (FP + TN + 1e-10) if (FP + TN) > 0 else 0,
            'FNR': FN / (FN + TP + 1e-10) if (FN + TP) > 0 else 0
        }
    return metrics
 
# Iterate over classifiers
for clf_name, clf in classifiers.items():
    fold_idx = 1
    for train_index, test_index in kf.split(X):
        # Split the data
        X_train, X_test = X[train_index], X[test_index]
        y_train, y_test = y[train_index], y[test_index]
 
        # Record start time
        start_train_time = time.time()
        clf.fit(X_train, y_train)
        train_time = time.time() - start_train_time
 
        start_test_time = time.time()
        y_pred = clf.predict(X_test)
        test_time = time.time() - start_test_time
 
        timing_results.append({
            'Classifier': clf_name,
            'Fold': fold_idx,
            'Training Time (s)': train_time,
            'Testing Time (s)': test_time,
            'Total Time (s)': train_time + test_time
        })
 
        # Compute metrics
        unique_classes = np.unique(np.concatenate((y_test, y_pred)))
        cm = confusion_matrix(y_test, y_pred, labels=unique_classes)
        cm_metrics = confusion_matrix_metrics(cm, unique_classes)
 
        class_metrics_list = []
 
        for class_label in unique_classes:
            class_mask = (y_test == class_label)
            class_pred_mask = (y_pred == class_label)
            
            # Binary classification metrics for this class
            y_true_binary = (y_test == class_label).astype(int)
            y_pred_binary = (y_pred == class_label).astype(int)
            
            if class_mask.sum() == 0:
                # Skip classes with no instances in the test set for this fold
                class_specific_metrics = {
                    'Classifier': clf_name,
                    'Fold': fold_idx,
                    'Class': class_label,
                    'Accuracy': np.nan,
                    'Precision': np.nan,
                    'Recall': np.nan,
                    'F1 Score': np.nan,
                    'Balanced Accuracy': np.nan,
                    'True Positive Rate (TPR)': np.nan,
                    'True Negative Rate (TNR)': np.nan,
                    'False Positive Rate (FPR)': np.nan,
                    'False Negative Rate (FNR)': np.nan,
                    'Training Time (s)': train_time,
                    'Testing Time (s)': test_time
                }
            else:
                class_specific_metrics = {
                    'Classifier': clf_name,
                    'Fold': fold_idx,
                    'Class': class_label,
                    'Accuracy': accuracy_score(y_true_binary, y_pred_binary),
                    'Precision': precision_score(y_true_binary, y_pred_binary, zero_division=0),
                    'Recall': recall_score(y_true_binary, y_pred_binary, zero_division=0),
                    'F1 Score': f1_score(y_true_binary, y_pred_binary, zero_division=0),
                    'Balanced Accuracy': balanced_accuracy_score(y_true_binary, y_pred_binary),
                    'True Positive Rate (TPR)': cm_metrics[class_label]['TPR'],
                    'True Negative Rate (TNR)': cm_metrics[class_label]['TNR'],
                    'False Positive Rate (FPR)': cm_metrics[class_label]['FPR'],
                    'False Negative Rate (FNR)': cm_metrics[class_label]['FNR'],
                    'Training Time (s)': train_time,
                    'Testing Time (s)': test_time
                }
 
            class_metrics_list.append(class_specific_metrics)
 
        # Plot and save confusion matrix
        plt.figure(figsize=(10, 8))
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=unique_classes, yticklabels=unique_classes)
        plt.title(f"{clf_name} - Fold {fold_idx} Confusion Matrix")
        plt.xlabel("Predicted")
        plt.ylabel("True")
        plt.savefig(f"confusion_matrices/{clf_name}_fold_{fold_idx}.png")
        plt.close()
 
        # Append results for this fold
        results.extend(class_metrics_list)
        fold_idx += 1
 
timing_df = pd.DataFrame(timing_results)
timing_df.to_csv("time_dataset2.csv", index=False)
 
# Create a DataFrame for results
results_df = pd.DataFrame(results)
print("Classification Metrics Across Folds:")
print(results_df)
 
# Save results to CSV
results_df.to_csv("metrics_dataset2.csv", index=False)

# Additional Fuzzy Logic Visualization
def visualize_fuzzy_membership():
    # Create universe variables
    x_range = np.linspace(0, 1, 100)
    
    # Create antecedents for each feature
    feature_0 = ctrl.Antecedent(x_range, 'feature_0')
    feature_1 = ctrl.Antecedent(x_range, 'feature_1')
    feature_2 = ctrl.Antecedent(x_range, 'feature_2')
    feature_3 = ctrl.Antecedent(x_range, 'feature_3')
    attack_class = ctrl.Consequent(np.linspace(1, 15, 100), 'attack_class')
    
    # Auto-membership functions
    feature_0.automf(3, names=['low', 'medium', 'high'])
    feature_1.automf(3, names=['low', 'medium', 'high'])
    feature_2.automf(3, names=['low', 'medium', 'high'])
    feature_3.automf(2, names=['low', 'high'])
    
    # Some representative attack classes
    attack_class['low_class'] = fuzz.trimf(attack_class.universe, [4, 6, 8])
    attack_class['mid_class'] = fuzz.trimf(attack_class.universe, [8, 10, 12])
    attack_class['high_class'] = fuzz.trimf(attack_class.universe, [10, 12, 14])
    attack_class['very_high_class'] = fuzz.trimf(attack_class.universe, [12, 14, 15])
    
    # Create a directory to save fuzzy membership plots
    os.makedirs("fuzzy_membership_dataset2", exist_ok=True)
    
    # Visualize membership functions
    plt.figure(figsize=(10, 4))
    feature_0.view()
    plt.title("Feature 0 Membership Functions")
    plt.savefig("fuzzy_membership_dataset2/feature_0_membership.png")
    plt.close()
    
    plt.figure(figsize=(10, 4))
    feature_1.view()
    plt.title("Feature 1 Membership Functions")
    plt.savefig("fuzzy_membership_dataset2/feature_1_membership.png")
    plt.close()
    
    plt.figure(figsize=(10, 4))
    feature_2.view()
    plt.title("Feature 2 Membership Functions")
    plt.savefig("fuzzy_membership_dataset2/feature_2_membership.png")
    plt.close()
    
    plt.figure(figsize=(10, 4))
    feature_3.view()
    plt.title("Feature 3 Membership Functions")
    plt.savefig("fuzzy_membership_dataset2/feature_3_membership.png")
    plt.close()
    
    plt.figure(figsize=(10, 4))
    attack_class.view()
    plt.title("Attack Class Membership Functions")
    plt.savefig("fuzzy_membership_dataset2/attack_class_membership.png")
    plt.close()

# Call the function to generate fuzzy membership plots
visualize_fuzzy_membership()

# Generate a summary of fuzzy logic performance
print("\nFuzzy Logic Classifier Summary:")
fuzzy_results = results_df[results_df['Classifier'] == 'FuzzyLogic'].copy()

# Calculate mean values for each class manually
class_summaries = {}
for class_label in fuzzy_results['Class'].unique():
    class_data = fuzzy_results[fuzzy_results['Class'] == class_label]
    class_summary = {
        'Accuracy': class_data['Accuracy'].mean(),
        'Precision': class_data['Precision'].mean(),
        'Recall': class_data['Recall'].mean(),
        'F1 Score': class_data['F1 Score'].mean(),
        'Balanced Accuracy': class_data['Balanced Accuracy'].mean(),
        'True Positive Rate (TPR)': class_data['True Positive Rate (TPR)'].mean(),
        'True Negative Rate (TNR)': class_data['True Negative Rate (TNR)'].mean(),
        'False Positive Rate (FPR)': class_data['False Positive Rate (FPR)'].mean(),
        'False Negative Rate (FNR)': class_data['False Negative Rate (FNR)'].mean(),
        'Training Time (s)': class_data['Training Time (s)'].mean(),
        'Testing Time (s)': class_data['Testing Time (s)'].mean()
    }
    class_summaries[class_label] = class_summary

# Convert to DataFrame for display and saving
fuzzy_summary = pd.DataFrame(class_summaries).T
print(fuzzy_summary)
fuzzy_summary.to_csv("fuzzy_summary_dataset2.csv")

# Generate a comparison of classifiers
print("\nClassifier Performance Comparison:")
# Create an empty DataFrame to store aggregated results
classifier_comparison = pd.DataFrame()

# Aggregate metrics for each classifier
for clf_name in results_df['Classifier'].unique():
    clf_data = results_df[results_df['Classifier'] == clf_name]
    
    # Calculate means for each metric
    agg_metrics = {
        'Accuracy': clf_data['Accuracy'].mean(),
        'Precision': clf_data['Precision'].mean(),
        'Recall': clf_data['Recall'].mean(),
        'F1 Score': clf_data['F1 Score'].mean(),
        'Balanced Accuracy': clf_data['Balanced Accuracy'].mean(),
        'Training Time (s)': clf_data['Training Time (s)'].mean(),
        'Testing Time (s)': clf_data['Testing Time (s)'].mean()
    }
    
    # Add to comparison DataFrame
    classifier_comparison.loc[clf_name] = agg_metrics

print(classifier_comparison)
classifier_comparison.to_csv("classifier_comparison_dataset2.csv")

# Plot classifier performance comparison
plt.figure(figsize=(12, 8))
metrics_to_plot = ['Accuracy', 'Precision', 'Recall', 'F1 Score', 'Balanced Accuracy']
classifier_comparison[metrics_to_plot].plot(kind='bar')
plt.title('Classifier Performance Comparison - Dataset 2')
plt.ylabel('Score')
plt.xlabel('Classifier')
plt.legend(loc='lower right')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.savefig('classifier_comparison_dataset2.png')
plt.close()

# Plot training and testing times
plt.figure(figsize=(10, 6))
timing_comparison = timing_df.groupby('Classifier').agg({
    'Training Time (s)': 'mean',
    'Testing Time (s)': 'mean',
    'Total Time (s)': 'mean'
})
timing_comparison.plot(kind='bar')
plt.title('Classifier Timing Comparison - Dataset 2')
plt.ylabel('Time (seconds)')
plt.xlabel('Classifier')
plt.legend()
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.savefig('timing_comparison_dataset2.png')
plt.close()

# Generate feature importance visualization for Decision Tree
dt_classifier = DecisionTreeClassifier()
dt_classifier.fit(X, y)

# Get feature importance
feature_importance = dt_classifier.feature_importances_
feature_names = df.columns[:-1]  # All columns except the last one (label)

# Sort feature importance
sorted_idx = np.argsort(feature_importance)
plt.figure(figsize=(12, 8))
plt.barh(range(len(sorted_idx)), feature_importance[sorted_idx], align='center')
plt.yticks(range(len(sorted_idx)), feature_names[sorted_idx])
plt.title('Feature Importance - Decision Tree')
plt.xlabel('Importance')
plt.tight_layout()
plt.savefig('feature_importance_dataset2.png')
plt.close()

Classification Metrics Across Folds:
       Classifier  Fold  Class  Accuracy  Precision    Recall  F1 Score  \
0    DecisionTree     1      0  0.999954   0.000000  0.000000  0.000000   
1    DecisionTree     1      1  0.995156   0.902672  0.895833  0.899240   
2    DecisionTree     1      2  0.999863   0.000000  0.000000  0.000000   
3    DecisionTree     1      3  0.999817   0.000000  0.000000  0.000000   
4    DecisionTree     1      4  0.999452   0.954545  0.954545  0.954545   
..            ...   ...    ...       ...        ...       ...       ...   
261    FuzzyLogic     2     29  0.998263   0.000000  0.000000  0.000000   
262    FuzzyLogic     2     30  0.999954   0.000000  0.000000  0.000000   
263    FuzzyLogic     2     31  0.999909   0.000000  0.000000  0.000000   
264    FuzzyLogic     2     32  0.999360   0.000000  0.000000  0.000000   
265    FuzzyLogic     2     33  0.999909   0.000000  0.000000  0.000000   

     Balanced Accuracy  True Positive Rate (TPR)  True Negativ

/home/yashbuddhadev21/.local/lib/python3.10/site-packages/skfuzzy/control/fuzzyvariable.py:125: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig.show()



Fuzzy Logic Classifier Summary:
    Accuracy  Precision  Recall  F1 Score  Balanced Accuracy  \
0   0.999954   0.000000     0.0  0.000000           0.500000   
1   0.976417   0.000000     0.0  0.000000           0.500000   
2   0.999954   0.000000     0.0  0.000000           0.500000   
3   0.999909   0.000000     0.0  0.000000           0.500000   
4   0.993670   0.000000     0.0  0.000000           0.500000   
5   0.999589   0.000000     0.0  0.000000           0.500000   
6   0.845365   0.000000     0.0  0.000000           0.500000   
7   0.989420   0.000000     0.0  0.000000           0.500000   
8   0.125820   0.090789     1.0  0.166462           0.521106   
9   0.912456   0.000000     0.0  0.000000           0.499962   
10  0.871232   0.000000     0.0  0.000000           0.478898   
11  0.999634   0.000000     0.0  0.000000           0.500000   
12  0.922899   0.000000     0.0  0.000000           0.499963   
13  0.905212   0.000000     0.0  0.000000           0.500000   
14  0.8

ValueError: cannot set a frame with no defined columns

<Figure size 1000x400 with 0 Axes>

<Figure size 1000x400 with 0 Axes>

<Figure size 1000x400 with 0 Axes>

<Figure size 1000x400 with 0 Axes>

<Figure size 1000x400 with 0 Axes>